# Exercise 5: Conditional Routing in LangGraph

**Level:** Basic

Linear graphs are useful but limited. Real agents need to **branch** based on conditions and **loop** until quality criteria are met. This exercise covers conditional edges — the core mechanism that makes LangGraph powerful.

**What you will learn:**
- `add_conditional_edges` for dynamic routing
- Writing route functions that return node names
- Testing different execution paths
- Building loops with a `should_continue` pattern

## 1. Setup & Installation

In [ ]:
!pip install langgraph langchain langchain-groq -q

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. Conditional Edges — The Basics

A **conditional edge** calls a routing function that inspects the current state and returns the name of the next node to execute.

```python
def route_function(state) -> str:
    if some_condition(state):
        return "node_a"
    else:
        return "node_b"

builder.add_conditional_edges("source_node", route_function)
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class RequestState(TypedDict):
    request_type: str    # "booking", "complaint", "info"
    message: str
    response: str

# Node functions
def classify_request(state: RequestState) -> dict:
    """Classify the request based on keywords."""
    msg = state["message"].lower()
    if any(word in msg for word in ["book", "reserve", "buy", "ticket"]):
        req_type = "booking"
    elif any(word in msg for word in ["complaint", "problem", "issue", "angry", "refund"]):
        req_type = "complaint"
    else:
        req_type = "info"
    print(f"  [classify] Classified as: {req_type}")
    return {"request_type": req_type}

def handle_booking(state: RequestState) -> dict:
    print("  [booking] Processing booking request")
    return {"response": f"Booking handler: Processing your booking request for '{state['message']}'"}

def handle_complaint(state: RequestState) -> dict:
    print("  [complaint] Escalating complaint")
    return {"response": f"Complaint handler: We're sorry. Escalating your issue regarding '{state['message']}'"}

def handle_info(state: RequestState) -> dict:
    print("  [info] Providing information")
    return {"response": f"Info handler: Here's information about '{state['message']}'"}

# Route function — returns the next node name
def route_request(state: RequestState) -> str:
    """Route to the appropriate handler based on request type."""
    return state["request_type"]

# Build the graph
builder = StateGraph(RequestState)

builder.add_node("classify", classify_request)
builder.add_node("booking", handle_booking)
builder.add_node("complaint", handle_complaint)
builder.add_node("info", handle_info)

builder.add_edge(START, "classify")

# Conditional edge: after classification, route to the right handler
builder.add_conditional_edges(
    "classify",           # Source node
    route_request,         # Route function
    {                      # Mapping: return value → node name
        "booking": "booking",
        "complaint": "complaint",
        "info": "info"
    }
)

# All handlers go to END
builder.add_edge("booking", END)
builder.add_edge("complaint", END)
builder.add_edge("info", END)

router_graph = builder.compile()
print("Router graph compiled!")

In [ ]:
# Visualize
print(router_graph.get_graph().draw_mermaid())

## 3. Testing Different Paths

In [ ]:
# Test 1: Booking request
print("=== Test 1: Booking ===")
result = router_graph.invoke({"message": "I want to book a flight to Istanbul"})
print(f"Response: {result['response']}\n")

# Test 2: Complaint
print("=== Test 2: Complaint ===")
result = router_graph.invoke({"message": "I have a complaint about my delayed flight, I want a refund"})
print(f"Response: {result['response']}\n")

# Test 3: Info request
print("=== Test 3: Info ===")
result = router_graph.invoke({"message": "What are your destinations in Europe?"})
print(f"Response: {result['response']}")

## 4. LLM-Based Routing

Keyword matching is brittle. Let's use an LLM to classify requests more intelligently.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

class Classification(BaseModel):
    """Request classification."""
    category: Literal["booking", "complaint", "info"] = Field(
        description="The category of the customer request"
    )
    confidence: float = Field(description="Confidence score 0-1")

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
classifier = model.with_structured_output(Classification)

class SmartRequestState(TypedDict):
    message: str
    category: str
    confidence: float
    response: str

def llm_classify(state: SmartRequestState) -> dict:
    """Use an LLM to classify the request."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Classify the customer request into: booking, complaint, or info."),
        ("human", "{message}")
    ])
    chain = prompt | classifier
    result = chain.invoke({"message": state["message"]})
    print(f"  [llm_classify] {result.category} (confidence: {result.confidence})")
    return {"category": result.category, "confidence": result.confidence}

def smart_booking(state: SmartRequestState) -> dict:
    return {"response": f"Booking: We'll help you book. Your request: '{state['message']}'"}

def smart_complaint(state: SmartRequestState) -> dict:
    return {"response": f"Complaint: We apologize. Escalating: '{state['message']}'"}

def smart_info(state: SmartRequestState) -> dict:
    return {"response": f"Info: Here's what you need to know about: '{state['message']}'"}

def smart_route(state: SmartRequestState) -> str:
    return state["category"]

builder = StateGraph(SmartRequestState)
builder.add_node("classify", llm_classify)
builder.add_node("booking", smart_booking)
builder.add_node("complaint", smart_complaint)
builder.add_node("info", smart_info)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", smart_route, {
    "booking": "booking",
    "complaint": "complaint",
    "info": "info"
})
builder.add_edge("booking", END)
builder.add_edge("complaint", END)
builder.add_edge("info", END)

smart_graph = builder.compile()
print("Smart router compiled!")

In [ ]:
# Test with ambiguous/natural language
test_messages = [
    "I'd love to fly to Barcelona next month",
    "Your service was terrible, my luggage was lost and nobody helped me!",
    "What's the baggage allowance for economy class?",
    "Can I upgrade my seat on the flight tomorrow?",
]

for msg in test_messages:
    print(f"\nInput: {msg}")
    result = smart_graph.invoke({"message": msg})
    print(f"Category: {result['category']} (confidence: {result['confidence']})")
    print(f"Response: {result['response']}")

## 5. Loops — The `should_continue` Pattern

One of the most powerful patterns in agent development: **loop until a quality condition is met**. This is how agents self-correct.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator
import random

class RefineState(TypedDict):
    topic: str
    draft: str
    quality_score: float
    iteration: int
    feedback_history: Annotated[list[str], operator.add]

def write_draft(state: RefineState) -> dict:
    """Write or rewrite the draft."""
    iteration = state.get("iteration", 0) + 1
    
    if iteration == 1:
        draft = f"First draft about {state['topic']}: This is a basic article."
    else:
        draft = f"Revision {iteration} about {state['topic']}: Improved based on feedback."
    
    print(f"  [write] Iteration {iteration}: wrote draft")
    return {"draft": draft, "iteration": iteration}

def evaluate_quality(state: RefineState) -> dict:
    """Evaluate the quality of the draft."""
    # Simulate quality improving with each iteration
    base_score = 0.3 + (state["iteration"] * 0.2)
    score = min(base_score + random.uniform(-0.1, 0.1), 1.0)
    
    if score < 0.7:
        feedback = f"Score {score:.2f}: Needs more depth and better structure."
    else:
        feedback = f"Score {score:.2f}: Quality meets the threshold!"
    
    print(f"  [evaluate] {feedback}")
    return {
        "quality_score": score,
        "feedback_history": [feedback]
    }

def should_continue(state: RefineState) -> str:
    """Decide whether to continue refining or finish."""
    if state["quality_score"] >= 0.7:
        print(f"  [route] Quality {state['quality_score']:.2f} >= 0.7 → DONE")
        return "done"
    if state["iteration"] >= 5:
        print(f"  [route] Max iterations reached → DONE")
        return "done"
    print(f"  [route] Quality {state['quality_score']:.2f} < 0.7 → REWRITE")
    return "rewrite"

# Build the graph with a loop
builder = StateGraph(RefineState)

builder.add_node("write", write_draft)
builder.add_node("evaluate", evaluate_quality)

builder.add_edge(START, "write")
builder.add_edge("write", "evaluate")

# Conditional: evaluate → write (loop) or evaluate → END
builder.add_conditional_edges(
    "evaluate",
    should_continue,
    {
        "rewrite": "write",   # Loop back
        "done": END            # Exit
    }
)

refine_graph = builder.compile()
print("Refinement graph compiled!")

In [ ]:
# Run the refinement loop
print("Running refinement loop...\n")

result = refine_graph.invoke({
    "topic": "AI in Aviation",
    "draft": "",
    "quality_score": 0.0,
    "iteration": 0,
    "feedback_history": []
})

print(f"\nFinal result:")
print(f"  Iterations: {result['iteration']}")
print(f"  Final score: {result['quality_score']:.2f}")
print(f"  Final draft: {result['draft']}")
print(f"  Feedback history:")
for fb in result['feedback_history']:
    print(f"    - {fb}")

In [ ]:
# Visualize — notice the loop!
print(refine_graph.get_graph().draw_mermaid())

## 6. LLM-Powered Refinement Loop

Now let's build the same pattern but with actual LLM calls for writing and evaluating.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

class QualityEval(BaseModel):
    score: float = Field(description="Quality score from 0.0 to 1.0")
    feedback: str = Field(description="Specific feedback for improvement")

class LLMRefineState(TypedDict):
    topic: str
    draft: str
    score: float
    feedback: str
    iteration: int

def llm_write(state: LLMRefineState) -> dict:
    iteration = state.get("iteration", 0) + 1
    
    if iteration == 1:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Write a short paragraph (3-4 sentences) about the given topic."),
            ("human", "Topic: {topic}")
        ])
        chain = prompt | model | StrOutputParser()
        draft = chain.invoke({"topic": state["topic"]})
    else:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Rewrite and improve this paragraph based on the feedback. Keep it to 3-4 sentences."),
            ("human", "Draft:\n{draft}\n\nFeedback:\n{feedback}\n\nWrite an improved version.")
        ])
        chain = prompt | model | StrOutputParser()
        draft = chain.invoke({"draft": state["draft"], "feedback": state["feedback"]})
    
    print(f"  [write] Iteration {iteration}")
    return {"draft": draft, "iteration": iteration}

def llm_evaluate(state: LLMRefineState) -> dict:
    evaluator = model.with_structured_output(QualityEval)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Evaluate this paragraph for clarity, engagement, and accuracy. "
                    "Be critical. Only give scores above 0.8 for genuinely excellent writing."),
        ("human", "Evaluate this:\n{draft}")
    ])
    chain = prompt | evaluator
    result = chain.invoke({"draft": state["draft"]})
    print(f"  [evaluate] Score: {result.score}, Feedback: {result.feedback[:80]}...")
    return {"score": result.score, "feedback": result.feedback}

def llm_should_continue(state: LLMRefineState) -> str:
    if state["score"] >= 0.8:
        return "done"
    if state["iteration"] >= 3:
        return "done"
    return "rewrite"

builder = StateGraph(LLMRefineState)
builder.add_node("write", llm_write)
builder.add_node("evaluate", llm_evaluate)

builder.add_edge(START, "write")
builder.add_edge("write", "evaluate")
builder.add_conditional_edges("evaluate", llm_should_continue, {
    "rewrite": "write",
    "done": END
})

llm_refine_graph = builder.compile()
print("LLM refinement graph compiled!")

In [ ]:
# Run the LLM refinement loop
print("Running LLM refinement...\n")

result = llm_refine_graph.invoke({
    "topic": "How AI agents are changing the airline industry",
    "draft": "",
    "score": 0.0,
    "feedback": "",
    "iteration": 0
})

print(f"\n{'='*60}")
print(f"Iterations: {result['iteration']}")
print(f"Final score: {result['score']}")
print(f"Final draft:\n{result['draft']}")

---
## YOUR TURN: Exercise A

Build a **support ticket router** with conditional edges:

1. `intake` node — receives the ticket text
2. `classify` node — uses LLM to classify as `urgent`, `normal`, or `low`
3. Route based on classification:
   - `urgent` → `escalate` node (immediate response)
   - `normal` → `respond` node (standard response)
   - `low` → `auto_reply` node (template response)
4. All paths end at a `log` node that records the outcome

Test with at least 3 different tickets.

In [ ]:
# YOUR TURN: Build the support ticket router

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# TODO: Define TicketState

# TODO: Define node functions (intake, classify, escalate, respond, auto_reply, log)

# TODO: Define the route function

# TODO: Build and compile the graph

# TODO: Test with different ticket types:
# - "URGENT: System is down, customers cannot book flights!"
# - "I'd like to change my seat assignment for next week's flight."
# - "Just wondering, do you offer vegetarian meals?"

---
## YOUR TURN: Exercise B

Build a **quiz generator with retry logic**:

1. `generate` node — LLM generates a quiz question about a given topic
2. `validate` node — LLM checks if the question is clear, unambiguous, and has a definitive answer
3. If validation fails → loop back to `generate` with feedback
4. If validation passes → go to `format` node that outputs the final question
5. Max 3 attempts before giving up

This is a critical agent pattern: generate → validate → retry.

In [ ]:
# YOUR TURN: Build the quiz generator with retry

# TODO: Define state (topic, question, is_valid, feedback, attempt_count)

# TODO: Define generate, validate, and format nodes

# TODO: Define the should_continue route function

# TODO: Build the graph with the loop

# TODO: Test with topics like "Python programming", "World Geography"

## Key Takeaways

- **Conditional edges** let you route dynamically: `add_conditional_edges(source, route_fn, mapping)`
- The **route function** inspects state and returns the next node's name
- **Loops** are created by routing back to an earlier node (e.g., `evaluate → write`)
- Always add a **max iteration guard** to prevent infinite loops
- The `should_continue` pattern is fundamental: generate → evaluate → continue or stop
- LLM-based routing is more flexible than keyword matching but costs API calls

**Next:** In Exercise 6, we will build a multi-agent system where multiple specialized agents collaborate through a LangGraph.